In [ ]:
import numpy as np
import anndata as an
import scanpy as sc
import scipy
import os
import torch
import pandas as pd
import decoupler as dc
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

In [ ]:
genes = sc.read_h5ad('reheatHeart/reheatHeart.h5ad',backed='r').var_names
genes = np.array(genes)

In [ ]:
qs = torch.load('results/heart/qs.pt',map_location='cpu')
qs = np.array([x.detach().numpy() for x in qs])
#qs = qs.transpose(1, 0, 2).reshape(2000, 20)
#qs = qs[4][:,2]
#qs

In [ ]:
markers = dc.op.resource("PanglaoDB", organism="human")
markers = markers[
    markers["human"].astype(bool)
    & markers["canonical_marker"].astype(bool)
    & (markers["human_sensitivity"].astype(float) > 0.5)
]

markers = markers[~markers.duplicated(["cell_type", "genesymbol"])]

markers = markers.rename(columns={"cell_type": "source", "genesymbol": "target"})
markers = markers[["source", "target"]]
markers

In [ ]:
markers['source'].unique()

In [ ]:
from IPython.display import display
for i in range(6):
    df = pd.DataFrame([qs[0][:,i].T], columns=genes,  index=[f'cluster_{i}'])
    tf_acts, tf_padj = dc.mt.ulm(data=df, net=markers)
    msk = (tf_padj.T < 0.05).iloc[:, 0]
    tf_acts = tf_acts.loc[:, msk]
    display(tf_acts)

In [ ]:
for i in range(6):
    df = pd.DataFrame([qs[1][:,i].T], columns=genes,  index=[f'cluster_{i}'])
    tf_acts, tf_padj = dc.mt.ulm(data=df, net=markers)
    msk = (tf_padj.T < 0.05).iloc[:, 0]
    tf_acts = tf_acts.loc[:, msk]
    display(tf_acts)

In [ ]:
for i in range(6):
    df = pd.DataFrame([qs[2][:,i].T], columns=genes,  index=[f'cluster_{i}'])
    tf_acts, tf_padj = dc.mt.ulm(data=df, net=markers)
    msk = (tf_padj.T < 0.05).iloc[:, 0]
    tf_acts = tf_acts.loc[:, msk]
    display(tf_acts)

In [ ]:
for i in range(6):
    df = pd.DataFrame([qs[3][:,i].T], columns=genes,  index=[f'cluster_{i}'])
    tf_acts, tf_padj = dc.mt.ulm(data=df, net=markers)
    msk = (tf_padj.T < 0.05).iloc[:, 0]
    tf_acts = tf_acts.loc[:, msk]
    display(tf_acts)

In [ ]:
for i in range(6):
    df = pd.DataFrame([qs[4][:,i].T], columns=genes,  index=[f'cluster_{i}'])
    tf_acts, tf_padj = dc.mt.ulm(data=df, net=markers)
    msk = (tf_padj.T < 0.05).iloc[:, 0]
    tf_acts = tf_acts.loc[:, msk]
    display(tf_acts)

In [ ]:
file_paths = [
    'reheatHeart/mrkrs/Chaffin_2022.csv',
    'reheatHeart/mrkrs/Kuppe_2022.csv',
    'reheatHeart/mrkrs/Koenig_2022.csv',
    'reheatHeart/mrkrs/Reichart_2022.csv',
    'reheatHeart/mrkrs/Simonson_2023.csv'
]

dfs = [pd.read_csv(f) for f in file_paths]
for i in range(len(dfs)):
    dfs[i] = dfs[i][(dfs[i]['FDR'] < 0.01) & (dfs[i]['logFC'] > 2)]

In [ ]:
mrkrs_by_study = []
for df in dfs:
    gene_by_name = dfs[0].groupby('name')['gene'].apply(list).to_dict()
    mrkrs_by_study.append(gene_by_name)
#mrkrs_by_study[0] :  chaffin: {lymphoid ['KIAA0355', 'GENE2', 'GENE3', ...]
                     #myeloid ['MYH7', 'GENE5', 'GENE6', ...]
                     #.....}

In [ ]:
studys = [np.argmax(q, axis=1) for q in qs] #chaffin, kuppe, koenig, reichart, simonson
clusters = [[0,1,2,3,4,5],[3,5,2,0,4,1],[2,3,1,4,5,0],[5,2,4,0,3,1],[0,4,2,5,3,1]]
genes_by_study = []
for i in range(len(clusters)):
    genes_by_study.append(
         {
            'lymphoid': genes[np.where(studys[i]==clusters[i][0])[0]],
            'myeloid': np.concatenate([genes[np.where(studys[i]==clusters[i][1])[0]],
                                      genes[np.where(studys[i]==clusters[i][4])[0]]]),
            'mesenchymal': np.concatenate([genes[np.where(studys[i]==clusters[i][2])[0]],
                                          genes[np.where(studys[i]==clusters[i][5])[0]]]),
            'endothelial': genes[np.where(studys[i]==clusters[i][3])[0]]
             
         }
     )
#genes_by_study[0] :  chaffin: {lymphoid ['KIAA0355', 'GENE2', 'GENE3', ...]
                     #myeloid {'MYH7', 'GENE5', 'GENE6', ...}
                     #.....}

In [ ]:
def fishers_test(study1,study2):
    fisher_lk = []
    ps = []
    for i in range(5):#5 studies
        marker = mrkrs_by_study[i]
        gene_module = genes_by_study[i]
        for cell_type in marker.keys():
            
            genes1 = set(marker[cell_type]) &set(genes)      # keep only markers that also belong to those 2000 genes
            genes2 = set(gene_module[cell_type])
            all_genes = set(genes)
            print(cell_type)
            a = len(genes1 & genes2)                         # both studies have gene 
            b = len(genes1 - genes2)                         # only study1
            c = len(genes2 - genes1)                         # only study2
            d = len(all_genes - (genes1 | genes2))           # neither
            
            table = [[a, b],
                     [c, d]]
            print(table)
            _, p = fisher_exact(table, alternative = 'greater')
            ps.append(p)
    _, ps_adj, _, _ = multipletests(ps, alpha=0.05, method='fdr_bh')
    
    return ps_adj 

In [ ]:
ps = fishers_test(mrkrs_by_study,genes_by_study)
ps

In [ ]:
cell_types = ['T cell', 'B cell', 'Macrophage', 'Endothelial']
study_names = ['chaffin', 'kuppe', 'koenig', 'reichart', 'simonson']
ps = -np.log10(ps)

plt.figure(figsize=(6,4))
plt.bar(cell_types, log_p, color='skyblue', edgecolor='black')
plt.ylabel('-log10(p-value)')
plt.title('Fisher Test Significance for Each Cell Type')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='p = 0.05')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
len(set(genes_by_study[0]['lymphoid']) & set(mrkrs_by_study[0]['myeloid']))